# 01 - NASA Battery Dataset Cleaning for Pacemaker RUL

This notebook adapts NASA Li-ion 18650 degradation data for Smart TwinPac pacemaker Remaining Useful Life (RUL) prediction.

**Physics adaptation**

- NASA data: aggressive charge/discharge cycles, approximately hours per cycle.
- Pacemaker target: continuous discharge, no recharge, slow degradation over 7 years.
- Adaptation: keep discharge only, remap cycle index to months, normalize/interpolate to 37 C body temperature, calculate RUL until voltage crosses 2.75 V, then augment sequences.

## 1. Load and inspect

Place NASA files in:

```text
data/raw/nasa_battery/B0005.mat
data/raw/nasa_battery/B0006.mat
data/raw/nasa_battery/B0007.mat
data/raw/nasa_battery/B0018.mat
```

The expected MATLAB structure is similar to:

```python
mat = scipy.io.loadmat('B0005.mat')
mat['B0005']['cycle'][0,0]['data']
```

The script extracts voltage, current, temperature, and capacity from each cycle.

In [ ]:
from pathlib import Path
import json
import scipy.io
import pandas as pd
import numpy as np

from backend.ml.prepare_nasa_battery_dataset import (
    PreparationConfig,
    inspect_mat_file,
    prepare_dataset,
)

RAW_DIR = Path('data/raw/nasa_battery')
BATTERIES = ['B0005', 'B0006', 'B0007', 'B0018']
RAW_DIR

In [ ]:
# Inspect available files before running the full pipeline.
available = []
for battery_id in BATTERIES:
    path = RAW_DIR / f'{battery_id}.mat'
    if path.exists():
        available.append(inspect_mat_file(path, battery_id))

available

## 2. Filter discharge only

The cleaning script keeps only discharge phases using three signals:

- NASA cycle type equals `discharge`.
- Or median current is negative.
- Or voltage is decreasing across the cycle.

For model output, current is normalized as discharge current with `current_a <= 0`, so validation can reject charge phases.

## 3. Remap cycles to pacemaker months

NASA cycles are accelerated lab cycles. The prototype remaps discharge-cycle progress to an 84-month pacemaker lifetime:

```python
month = cycle_id * (pacemaker_lifetime_months / nasa_total_cycles)
```

Example: 167 discharge cycles become approximately 84 months / 7 years.

## 4. Temperature normalization

NASA batteries may be measured away from body temperature. Smart TwinPac targets body temperature conditions, so voltage is normalized to 37 C using a configurable linear coefficient. The output sequence temperature is constrained to 35 C - 39 C after augmentation.

This is a prototype normalization step, not a full electrochemical temperature model.

## 5. Calculate RUL

RUL is calculated as days until the sequence reaches the prototype end-of-life threshold:

```python
rul_days = (end_month - current_month) * 30
```

The EOL threshold is `voltage <= 2.75 V`. RUL is clipped so it is never negative.

## 6. Augmentation

To increase LSTM training samples, the pipeline generates 500+ sequence variants using:

- Gaussian voltage noise: +/- 15 mV standard deviation.
- Gaussian temperature noise: +/- 0.5 C standard deviation.
- Time warping: stretch/compress discharge curves by +/- 10%.

Each sequence keeps a monthly axis and is grouped by `sequence_id`.

## 7. Train/test split and export

The split is done by `sequence_id`, not by individual rows, to avoid leakage between train and test sequences.

In [ ]:
config = PreparationConfig(
    raw_dir='data/raw/nasa_battery',
    output_train_csv='data/battery_train.csv',
    output_test_csv='data/battery_test.csv',
    output_stats_json='data/battery_stats.json',
    batteries=BATTERIES,
    primary_battery='B0005',
    pacemaker_lifetime_months=84.0,
    days_per_month=30.0,
    body_temp_c=37.0,
    body_temp_tolerance_c=2.0,
    voltage_bol_v=3.7,
    voltage_eol_v=2.75,
    temp_voltage_coeff_v_per_c=-0.003,
    total_sequences=500,
    test_fraction=0.2,
    random_seed=42,
    voltage_noise_std_v=0.015,
    temp_noise_std_c=0.5,
    max_time_warp_fraction=0.10,
)

train_df, test_df, stats = prepare_dataset(config)
train_df.head()

In [ ]:
test_df.head()

## Validation checks

The pipeline enforces:

- Voltage range: 2.7 V to 3.7 V.
- No charge phases: `current_a <= 0`.
- Temperature: 35 C to 39 C only.
- RUL never negative.
- Sequence length at least 30 months.

In [ ]:
stats['validation_checks']

## Metrics to report

Report these values in the thesis/prototype documentation:

- Total sequences.
- Average sequence length in months.
- Voltage correlation with RUL as R2.
- Temperature variance sigma.

In [ ]:
report = {
    'total_sequences': stats['total_sequences'],
    'train_sequences': stats['train_sequences'],
    'test_sequences': stats['test_sequences'],
    'avg_sequence_length_months': stats['avg_sequence_length_months'],
    'voltage_correlation_with_rul_r2': stats['voltage_correlation_with_rul_r2'],
    'temperature_variance_sigma': stats['temperature_variance_sigma'],
}
report

In [ ]:
print('Saved outputs:')
print('- data/battery_train.csv')
print('- data/battery_test.csv')
print('- data/battery_stats.json')

with open('data/battery_stats.json', 'r', encoding='utf-8') as f:
    saved_stats = json.load(f)
saved_stats['csv_schema']